# Pipeline Completo — Classificação de Exoplanetas

Notebook orquestrador que executa todas as 5 etapas do pipeline importando os módulos de `pipeline/`:

1. **Pré-processamento** — limpeza, remoção de outliers, PCA, ranges terrestres
2. **Hierarchical Clustering** — Ward, 4 clusters, análise de proximidade à Terra
3. **Label Propagation** — pseudo-rótulos TT/NT semi-supervisionados
4. **SVM** — classificação supervisionada (kernel RBF, split 75/15/10)
5. **Similaridade Terrestre** — índice 0-100% por planeta

> **Observação:** todas as figuras são salvas em `../outputs/figs/` e também exibidas inline.


> **Nota sobre execução:** este notebook detecta automaticamente se está rodando no Google Colab ou localmente. No Colab, monta seu Google Drive e assume que a pasta `exoplanetas/` está em `/content/drive/MyDrive/exoplanetas`. Ajuste o caminho na célula abaixo se necessário.

In [ ]:
# ─── Setup: detecta Colab e configura ambiente ────────────────
# Esta célula funciona tanto em Jupyter local quanto no Google Colab.
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # >>> AJUSTE o caminho do projeto no seu Drive se necessário <<<
    PROJECT_ROOT = Path('/content/drive/MyDrive/exoplanetas')
    # instala dependências que o Colab não tem por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'openpyxl'], check=False)
else:
    # Local: assume que este notebook está em <projeto>/notebooks/
    PROJECT_ROOT = Path.cwd().parent

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em: {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Projeto:  {PROJECT_ROOT}')

# Imports do pipeline
from pipeline import (
    preprocessing,
    hierarchical_clustering,
    label_propagation,
    svm,
    similaridade_terra,
    diagnostics,
)

# Caminhos padrão
DATA = PROJECT_ROOT / "data" / "PSCompData.xlsx"
OUT = PROJECT_ROOT / "outputs"
FIGS = OUT / "figs"
OUT.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

print(f"Dataset:  {DATA}")
print(f"Saída:    {OUT}")

## Etapa 1 — Pré-processamento

Filtragem de colunas, remoção de linhas com incerteza > 50%, remoção de outliers pelo IQR, padronização (StandardScaler) e PCA (2 componentes).

In [ ]:
result_pp = preprocessing.main(
    caminho_entrada=str(DATA),
    caminho_saida=str(OUT / "DFHierarchicalClustering.xlsx"),
    caminho_rgjson=str(OUT / "ranges_terrestres.json"),
    save_dir=str(FIGS),
    show=True,   # exibe boxplots e PCA inline
)

## Etapa 2 — Hierarchical Clustering

Agrupamento hierárquico (Ward) formando 4 clusters, dendrograma, scatter no espaço PCA e cálculo da distância de cada cluster ao centro terrestre.

In [ ]:
result_hc = hierarchical_clustering.main(
    caminho_entrada=str(OUT / "DFHierarchicalClustering.xlsx"),
    caminho_ranges=str(OUT / "ranges_terrestres.json"),
    caminho_saida=str(OUT / "DFLabelPropagation.xlsx"),
    caminho_info_json=str(OUT / "info_proximidade.json"),
    save_dir=str(FIGS),
    show=True,
)

## Etapa 3 — Label Propagation

Cluster mais próximo do centro terrestre vira TT (1), mais distante vira NT (0), demais ficam como -1 (não rotulados) e o algoritmo propaga os rótulos com kernel knn (k=7).

In [ ]:
result_lp = label_propagation.main(
    caminho_entrada=str(OUT / "DFLabelPropagation.xlsx"),
    caminho_info_proximidade=str(OUT / "info_proximidade.json"),
    caminho_saida=str(OUT / "DFsvm.xlsx"),
    save_dir=str(FIGS),
    show=True,
)

## Etapa 4 — Support Vector Machine

SVM com kernel RBF, split 75/15/10 (treino/ajuste/teste) estratificado por classe. Avaliação em ambos os conjuntos, análise detalhada dos erros e exportação da tabela final.

In [ ]:
result_svm = svm.main(
    caminho_entrada=str(OUT / "DFsvm.xlsx"),
    caminho_saida_csv=str(OUT / "resultados_svm.csv"),
    save_dir=str(FIGS),
    show=True,
)

print(f"\n>>> Acurácia final: {result_svm['acuracia']:.1%}")

## Etapa 5 — Similaridade com a Terra

Índice ponderado 0-100% por planeta, no espaço padronizado, usando os mesmos scaler e df_sem_outliers da Etapa 1.

In [ ]:
# Precisamos anexar pl_name ao df_sem_outliers para exportar os nomes
df_sem_outliers = result_pp["df_sem_outliers"].assign(
    pl_name=result_pp["df_original"].loc[
        result_pp["df_sem_outliers"].index, "pl_name"
    ]
)

df_similaridade = similaridade_terra.main(
    df_sem_outliers=df_sem_outliers,
    scaler=result_pp["scaler"],
    colunas_numericas=result_pp["colunas_numericas"],
    caminho_saida_csv=str(OUT / "similaridade_exoplanetas_terra.csv"),
)

## Diagnóstico 1 — Circularidade Cluster → LP → SVM

A acurácia de 99.6% no teste do SVM precisa ser lida com cuidado. O SVM foi treinado com rótulos que o Label Propagation gerou a partir dos clusters hierárquicos — ou seja, a "verdade" que ele está aprendendo a reproduzir foi produzida pelo próprio pipeline. Se o SVM concordar quase perfeitamente com o LP, e o LP concordar quase perfeitamente com uma regra trivial baseada no cluster, então o SVM não está agregando informação além do agrupamento original.

O relatório abaixo compara três atribuições sobre o **dataset inteiro** (sem split):

- **Baseline trivial**: cada cluster recebe o rótulo majoritário do LP dentro dele.
- **Label Propagation**: coluna `label_lp`.
- **SVM**: previsão do modelo treinado.

Se as três concordarem ~100%, a acurácia do teste é essencialmente circular.

In [ ]:
from pipeline import diagnostics
import pandas as pd, json

df_svm_full = pd.read_excel(OUT / "DFsvm.xlsx", index_col=0)
info = json.load(open(OUT / "info_proximidade.json"))

_ = diagnostics.relatorio_circularidade(
    df_svm_full,
    result_svm["modelo"],
    info["cluster_mais_proximo"],
    info["cluster_mais_distante"],
)

## Diagnóstico 2 — Validação externa

A única forma de checar se as fronteiras TT/NT correspondem a algo físico é testar o modelo em corpos que ele **nunca viu no treino** e cuja natureza é bem conhecida. Aqui aplicamos o SVM ao Sistema Solar completo:

- **Esperado TT**: Terra (referência), Vênus (Earth-like), possivelmente Mercúrio e Marte (embora fora do range definido pelo pipeline).
- **Esperado NT**: Júpiter, Saturno, Urano, Netuno (gigantes gasosos/de gelo).

A coluna `|z|_max` mostra quão longe cada corpo está da distribuição de treino (após padronização). Valores acima de ~4 significam que o modelo está **extrapolando** e a previsão perde confiabilidade.

In [ ]:
resultado_ss = diagnostics.validacao_externa(
    result_svm["modelo"],
    result_pp["scaler"],
)

## Interpretação e próximos passos

Os dois diagnósticos juntos deixam três conclusões operacionais:

**1. Sobre a acurácia do 99.6%.** Ela mede quão bem o SVM reproduz o Label Propagation, não quão bem o pipeline identifica planetas terrestres. É uma métrica interna de consistência, não de validade externa. Para o relatório da IC, vale reportá-la como *"consistência SVM ↔ LP"* em vez de *"acurácia de classificação de terrestrialidade"*.

**2. Sobre a validação externa.** Se a Terra e Vênus saem NT, o pipeline aprendeu uma fronteira que não corresponde à intuição física de "planeta Earth-like". Isso não é bug — é uma descoberta sobre o que o modelo realmente aprendeu. Duas causas prováveis:

- O `rm_outliers` por IQR removeu exatamente a região do espaço onde a Terra vive (planetas com períodos ~centenas de dias e órbitas quase circulares são raros no dataset porque são difíceis de detectar). O dataset filtrado é dominado por hot jupiters e sub-Neptunes com períodos curtos.
- Consequentemente, o `scaler` foi ajustado numa distribuição enviesada, e a Terra fica com `|z|_max ~ 10` — fortemente out-of-distribution.

**3. Sugestões de experimento** (para uma versão futura do pipeline):

- Substituir o filtro IQR estatístico por um filtro físico (ex.: manter tudo com `pl_orbper < 10.000 dias` e `pl_rade < 20 R⊕`), preservando a região Earth-like.
- Repetir o ajuste do scaler apenas na região "candidata a TT" (planetas com raio < 2 R⊕) e ver se a Terra passa a cair na região TT.
- Reportar precision/recall por classe (o desbalanceamento 21% TT / 79% NT torna a acurácia global pouco informativa).
- Comparar os resultados do SVM com um KNN treinado nos mesmos rótulos LP — se a acurácia for parecida, confirma que o SVM não está agregando não-linearidade útil.

## Resumo dos artefatos gerados

In [ ]:
print("Arquivos em outputs/:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        rel = p.relative_to(OUT)
        print(f"  {rel}   ({p.stat().st_size:,} bytes)")